# 03 Model A: TF-IDF
Vocabulary fitted on train complaints, threshold chosen on validation pairs, one evaluation on test pairs.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 110)
from sklearn.metrics import classification_report
from src.data_loader import load_complaints, load_pairs
from src.similarity import paired_cosine, predict, select_threshold
from src.tfidf_model import TfidfModel
m = TfidfModel().fit(load_complaints('train').complaint_text.tolist())
print('vocabulary size:', m.vocabulary_size)

vocabulary size: 13657


In [2]:
score = lambda p: paired_cosine(m.encode(p.complaint_1.tolist()), m.encode(p.complaint_2.tolist()))
val, test = load_pairs('val'), load_pairs('test')
t, f1 = select_threshold(score(val), val.duplicate_label)
print(f'threshold {t:.4f}, validation F1 {f1:.3f}')

threshold 0.0376, validation F1 0.772


In [3]:
st = score(test)
print(classification_report(test.duplicate_label, predict(st, t), target_names=['not duplicate', 'duplicate'], digits=3))

               precision    recall  f1-score   support

not duplicate      0.799     0.540     0.644       750
    duplicate      0.653     0.864     0.744       750

     accuracy                          0.702      1500
    macro avg      0.726     0.702     0.694      1500
 weighted avg      0.726     0.702     0.694      1500



## Why TF-IDF struggles: the hardest duplicates share almost no words

In [4]:
test.assign(score=st)[test.duplicate_label == 1].nsmallest(3, 'score')[['complaint_1', 'complaint_2', 'score']]

,complaint_1,complaint_2,score
48,dead cattle near society main gate,The carcass of a buffalo in Madhuban Vihar has not been removed! Nobody from the corporation has visited f...,0.0
360,"Dear Sir, this is to inform you that an old building in Tapi Residency is in a dangerous condition. We hav...",dilapidated structure near cricket ground,0.0
574,Jalaram Vihar me park ki koi dekhbhaal nahi hoti. 6 din se koi nahi aaya. Buzurgon ne walk pe jaana band k...,overgrown garden girls' high school 4 days,0.0
